In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from scipy.stats import loguniform
import xgboost as xgb
from sklearn.model_selection import ParameterGrid

# PREPARACIÓN DE DATOS

### Helpers modificados

In [ ]:
def load_data(path):
    df = pd.read_csv(path)
    # Si la primera columna es un índice guardado, borrar
    if "Unnamed: 0" in df.columns[0]:
        df = df.drop(columns=df.columns[0])
    return df

def get_feature_types(X):
    # Excluir columnas administrativas
    exclude = ['split', 'target', 'default']
    cols = [c for c in X.columns if c not in exclude]
    
    cat_cols = X[cols].select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    num_cols = X[cols].select_dtypes(include=[np.number]).columns.tolist()
    return cat_cols, num_cols

def encode_categoricals_ohe(X_fit, X_transform_list, cat_cols):
    """
    X_fit: DataFrame usado para aprender las categorías (SOLO TRAIN).
    X_transform_list: Lista de DataFrames a transformar [df_labeled, df_unlabeled].
    """
    if not cat_cols:
        return [df.copy() for df in X_transform_list]

    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoder.fit(X_fit[cat_cols])
    
    feature_names = encoder.get_feature_names_out(cat_cols)
    
    transformed_dfs = []
    for X in X_transform_list:
        encoded_data = encoder.transform(X[cat_cols])
        df_encoded = pd.DataFrame(encoded_data, columns=feature_names, index=X.index)
        transformed_dfs.append(df_encoded)
        
    return transformed_dfs

def scale_numericals(X_fit, X_transform_list, num_cols, method="standard"):
    """
    X_fit: DataFrame usado para calcular media/std (SOLO TRAIN).
    """
    if not num_cols:
        return [df.copy() for df in X_transform_list]

    scaler = StandardScaler() if method == "standard" else MinMaxScaler()
    scaler.fit(X_fit[num_cols])
    
    transformed_dfs = []
    for X in X_transform_list:
        scaled_data = scaler.transform(X[num_cols])
        df_scaled = pd.DataFrame(scaled_data, columns=num_cols, index=X.index)
        transformed_dfs.append(df_scaled)
        
    return transformed_dfs

### Función principal adaptada

In [ ]:
def prepare_data_bias_aware(config, path_labeled, path_unlabeled):
    """
    Prepara datos para el Bias-Aware Framework respetando el split predefinido.
    """
    print("--- Cargando datos ---")
    df_labeled = load_data(path_labeled)
    X_unlabeled = load_data(path_unlabeled)
    
    target_col = config.get("target", "default")
    split_col = "split"

    # 1. Separar Features y Metadata en Labeled
    # Guardamos target y split aparte, no queremos transformarlos
    y_labeled = df_labeled[target_col].copy()
    split_labeled = df_labeled[split_col].copy()
    
    X_labeled = df_labeled.drop(columns=[target_col, split_col])
    
    # Asegurar que X_unlabeled tenga las mismas columnas que X_labeled (orden)
    # y descartar columnas extra que puedan venir en el csv de rechazados
    common_cols_labeled = [c for c in X_labeled.columns if c in X_unlabeled.columns]
    X_unlabeled = X_unlabeled[common_cols_labeled]

    # Al contrario: descartar columnas de X_labeled que no estén en X_unlabeled
    common_cols_unlabeled = [c for c in X_unlabeled.columns if c in X_labeled.columns]
    X_labeled = X_labeled[common_cols_unlabeled]

    # 2. Identificar tipos de columnas
    cat_cols, num_cols = get_feature_types(X_labeled)
    
    # 3. Definir el conjunto de "Aprendizaje" (Fitting Set)
    # IMPORTANTE: Solo usamos 'train' para fitear scalers y encoders para evitar data leakage
    train_idx = split_labeled == 'train'
    X_fit_num = X_labeled.loc[train_idx, num_cols]
    X_fit_cat = X_labeled.loc[train_idx, cat_cols]
    
    print(f"Features Numéricas: {len(num_cols)} | Categóricas: {len(cat_cols)}")

    # 4. Transformaciones Numéricas
    print("--- Transformando Numéricas ---")
    num_method = config.get("numeric_transform", "standard")
    
    # Transformamos Labeled y Unlabeled usando las reglas aprendidas en Train
    list_num_processed = scale_numericals(
        X_fit_num, 
        [X_labeled, X_unlabeled], 
        num_cols, 
        method=num_method
    )
    X_labeled_num, X_unlabeled_num = list_num_processed

    # 5. Transformaciones Categóricas
    print("--- Transformando Categóricas ---")
    cat_method = config.get("categorical_encoding", "ohe")
    
    if cat_method == "ohe":
        list_cat_processed = encode_categoricals_ohe(
            X_fit_cat, 
            [X_labeled, X_unlabeled], 
            cat_cols
        )
        X_labeled_cat, X_unlabeled_cat = list_cat_processed
    else:
        # Si usas binary o label encoding, necesitarías adaptar esa función similar a encode_categoricals_ohe
        # Por ahora asumo OHE como default robusto
        raise NotImplementedError("Por ahora solo OHE está adaptado para alineación garantizada.")

    # 6. Re-ensamblaje
    print("--- Ensamblando Final ---")
    X_labeled_final = pd.concat([X_labeled_num, X_labeled_cat], axis=1)
    X_unlabeled_final = pd.concat([X_unlabeled_num, X_unlabeled_cat], axis=1)
    
    # 7. Alineación de seguridad (Por si OHE generó columnas distintas)
    # Esto es vital: Los rechazados deben tener EXACTAMENTE las mismas columnas que los aceptados
    X_unlabeled_final = X_unlabeled_final.reindex(columns=X_labeled_final.columns, fill_value=0)
    
    # 8. Reintegrar Target y Split a df_labeled
    # La clase BiasAwareSelfLearning espera un DF completo con target y split
    df_labeled_processed = pd.concat([X_labeled_final, split_labeled, y_labeled], axis=1)
    
    # NOTA: No se hace SMOTE/Undersampling aquí. El framework de Bias requiere las distribuciones originales.
    
    return {
        "df_labeled": df_labeled_processed, # Tiene features + target + split
        "X_unlabeled": X_unlabeled_final,   # Tiene features alineadas
        "feature_names": X_labeled_final.columns.tolist(),
        "target_col": target_col,
        "split_col": split_col
    }


# Ejecución de la preparación de datos

### Validador Ingresos + Sin Historial

In [ ]:
# --- EJECUCIÓN PREPRARAR ---
config = {
    "target": "default",
    "numeric_transform": "standard",
    "categorical_encoding": "ohe"
}

# Rutas de archivos
path_acc_VI_SH = '../data/bias/base_merged_validador_sinHistorial_sin_quantiles.csv'
path_rej_VI_SH = '../data/bias/base_merged_validador_sinHistorial_sin_quantiles_rechazados.csv'

data_prepared_VI_SH = prepare_data_bias_aware(config, path_acc_VI_SH, path_rej_VI_SH)

### Independiente + Quanto

In [ ]:
# Rutas de archivos
path_acc_I_Q= '../data/bias/base_merged_quanto_independientes_sin_quantiles.csv'
path_rej_I_Q = '../data/bias/base_merged_quanto_independientes_sin_quantiles_rechazados.csv'

data_prepared_I_Q = prepare_data_bias_aware(config, path_acc_I_Q, path_rej_I_Q)

## Limpieza nans (si hay)

In [ ]:
# revisar NaNs de data_prepared["df_labeled"] y data_prepared["X_unlabeled"]
print("Aceptados:")
for col in data_prepared_VI_SH["df_labeled"].columns:
    n_missing = data_prepared_VI_SH["df_labeled"][col].isnull().sum()
    if n_missing > 0:
        print(f"Columna '{col}': {n_missing} valores faltantes")

print("---------------------------------")
print("Rechazados:")
for col in data_prepared_VI_SH["X_unlabeled"].columns:
    n_missing = data_prepared_VI_SH["X_unlabeled"][col].isnull().sum()
    if n_missing > 0:
        print(f"Columna '{col}': {n_missing} valores faltantes")


In [ ]:
# revisar NaNs de data_prepared["df_labeled"] y data_prepared["X_unlabeled"]
print("Aceptados:")
for col in data_prepared_I_Q["df_labeled"].columns:
    n_missing = data_prepared_I_Q["df_labeled"][col].isnull().sum()
    if n_missing > 0:
        print(f"Columna '{col}': {n_missing} valores faltantes")

print("---------------------------------")
print("Rechazados:")
for col in data_prepared_I_Q["X_unlabeled"].columns:
    n_missing = data_prepared_I_Q["X_unlabeled"][col].isnull().sum()
    if n_missing > 0:
        print(f"Columna '{col}': {n_missing} valores faltantes")


### Eliminar nans

In [ ]:
# drop todos los registros con nans
data_prepared_VI_SH["df_labeled"] = data_prepared_VI_SH["df_labeled"].dropna()
data_prepared_VI_SH["X_unlabeled"] = data_prepared_VI_SH["X_unlabeled"].dropna()

# drop todos los registros con nans
data_prepared_I_Q["df_labeled"] = data_prepared_I_Q["df_labeled"].dropna()
data_prepared_I_Q["X_unlabeled"] = data_prepared_I_Q["X_unlabeled"].dropna()

# Bias-Aware Self-Learning method

### Helpers

In [ ]:
# Bayesian Evaluation Framework
def bayesian_evaluation(model, X_accepts, y_accepts, X_rejects, prior_prob_model, n_iterations=100):
    """
    Implementación del Algoritmo 1: Bayesian Evaluation Framework.
    
    Args:
        model: El modelo 'f(X)' a evaluar (Strong Learner).
        X_accepts, y_accepts: Datos de evaluación observados (parte del Holdout/Test).
        X_rejects: Datos de rechazados no etiquetados (parte del Holdout/Test).
        prior_prob_model: Modelo previo (o weak learner) para estimar P(y|x) de los rechazados.
        n_iterations (j_max): Número de simulaciones de Montecarlo.
    
    Returns:
        float: Métrica Bayesiana Promedio (BM).
    """
    # Predicciones del modelo actual sobre ambos grupos
    pred_accepts = model.predict_proba(X_accepts)[:, 1]
    pred_rejects = model.predict_proba(X_rejects)[:, 1]
    
    # Probabilidades a priori para los rechazados (usando un modelo previo o el weak learner)
    rejects_priors = prior_prob_model.predict_proba(X_rejects)[:, 1]
    
    scores = []
    
    # Montecarlo loop
    for j in range(n_iterations):
        # Generar etiquetas simuladas para rechazados basadas en una Binomial(1, p)
        # y_r = binomial(1, P(y^r|X^r))
        y_rejects_simulated = np.random.binomial(1, rejects_priors)
        
        # Concatenar Accepts reales con Rejects simulados
        y_combined = np.concatenate([y_accepts, y_rejects_simulated])
        pred_combined = np.concatenate([pred_accepts, pred_rejects])
        
        # Calcular métrica (ej. AUC) para esta iteración
        iter_score = roc_auc_score(y_combined, pred_combined)
        scores.append(iter_score)
        
    # Retornar el promedio (Esperanza)
    return np.mean(scores)

In [ ]:
# Similarity-Based Filtering Step
def filtering_step(X_accepts, X_rejects, beta_l=0.05, beta_u=0.95):
    """
    Filtra los rechazados basándose en su similaridad con los aceptados.
    
    Args:
        X_accepts: Features de los aceptados.
        X_rejects: Features de los rechazados.
        beta_l: Percentil inferior (eliminar outliers).
        beta_u: Percentil superior (eliminar redundantes).
        
    Returns:
        X_rejects_filtered: DataFrame con los rechazados que pasan el filtro.
    """
    # 1. Crear dataset temporal para el modelo de similaridad
    # Etiqueta 1 = Aceptado, 0 = Rechazado
    X_sim = pd.concat([X_accepts, X_rejects])
    y_sim = np.concatenate([np.ones(len(X_accepts)), np.zeros(len(X_rejects))])
    
    # 2. Entrenar modelo de similaridad (Propensity Model)
    # Un Random Forest suele capturar bien las fronteras no lineales
    sim_model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    sim_model.fit(X_sim, y_sim)
    
    # 3. Predecir probabilidad de "ser aceptado" para los rechazados
    prob_similarity = sim_model.predict_proba(X_rejects)[:, 1]
    
    # 4. Calcular los cortes basados en los percentiles beta
    lower_cut = np.percentile(prob_similarity, beta_l * 100)
    upper_cut = np.percentile(prob_similarity, beta_u * 100)
    
    # 5. Filtrar
    mask = (prob_similarity >= lower_cut) & (prob_similarity <= upper_cut)
    X_rejects_filtered = X_rejects[mask].copy()
    
    print(f"Filtering: {len(X_rejects)} originales -> {len(X_rejects_filtered)} seleccionados para Self-Learning.")
    print(f"Se eliminaron los muy distintos (prob < {lower_cut:.3f}) y los muy iguales (prob > {upper_cut:.3f})")
    
    return X_rejects_filtered

### Clase principal

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV # Importación necesaria

class BiasAwareSelfLearning:
    def __init__(
        self,
        strong_learner_type="xgb",   # ← parámetro nuevo
        random_state=42,
        n_iter_search=3,
        cv_folds=3
    ):

        self.random_state = random_state
        self.n_iter_search = n_iter_search
        self.cv_folds = cv_folds
        self.strong_learner_type = strong_learner_type   # ← guardar tipo

        # Weak Learner Estático
        self.weak_learner = LogisticRegression(
            l1_ratio=0.5,
            penalty="elasticnet",
            solver="saga",
            random_state=random_state,
            max_iter=5000
        )

        self.calibrated_weak_learner = None

        # Strong Learner
        self.best_model = None

        # HPO XGBoost
        self.xgb_param_dist = {
            "n_estimators": [50, 100, 200],
            "max_depth": [3, 4, 5],
            "learning_rate": [0.01, 0.05, 0.1],
            "subsample": [ 0.8, 0.9, 1.0],
            "colsample_bytree": [ 0.8, 0.9, 1.0],
            "gamma": [0, 0.1, 0.5],
            "reg_alpha": [0, 0.1, 1]
        }
        
    def _fit_and_calibrate_once(self, X_train, y_train, X_calib, y_calib):
        """Entrena y calibra una sola vez con CV manual."""
        X_combined = pd.concat([X_train, X_calib], axis=0, ignore_index=True)
        y_combined = pd.concat([y_train, y_calib], axis=0, ignore_index=True)
        
        n_train = len(X_train)
        custom_cv = [(np.arange(0, n_train), np.arange(n_train, len(X_combined)))]
        
        calibrated_model = CalibratedClassifierCV(
            estimator=self.weak_learner,
            method='isotonic',
            cv=custom_cv
        )
        calibrated_model.fit(X_combined, y_combined)
        return calibrated_model

    def _optimize_strong_learner(self, X, y):
        """
        Entrena el Strong Learner realizando optimización de hiperparámetros.
        El modelo depende de self.strong_learner_type.
        """
        y = pd.Series(y).astype(int).values
        # ===============================
        # Selección del modelo
        # ===============================
        if self.strong_learner_type == "xgb":

            base_model = xgb.XGBClassifier(
                eval_metric="logloss",
                random_state=self.random_state
            )

            param_dist = self.xgb_param_dist


        elif self.strong_learner_type == "logit":

            base_model = LogisticRegression(
                max_iter=1000,
                random_state=self.random_state
            )

            param_dist = {
                "C": [0.01, 0.1, 1, 10],
                "penalty": ["l2"],
                "solver": ["lbfgs"]
            }


        elif self.strong_learner_type == "rf":

            base_model = RandomForestClassifier(
                random_state=self.random_state
            )

            param_dist = {
                "n_estimators": [100, 200, 300],
                "max_depth": [3, 5, 8, None],
                "min_samples_leaf": [1, 5, 10],
                "max_features": ["sqrt", "log2"]
            }

        else:
            raise ValueError("strong_learner_type debe ser: 'xgb', 'logit' o 'rf'")

        # ===============================
        # Hyperparameter optimization
        # ===============================
        search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=param_dist,
            n_iter=self.n_iter_search,
            scoring="roc_auc",
            cv=self.cv_folds,
            random_state=self.random_state,
            n_jobs=-1,
            verbose=0
        )

        search.fit(X, y)

        print(f"    [HPO] Strong learner: {self.strong_learner_type}")
        print(f"    [HPO] Mejores params: {search.best_params_}")
        print(f"    [HPO] AUC en CV: {search.best_score_:.4f}")

        return search.best_estimator_

    def fit(self, df_labeled, X_unlabeled, target_col, split_col, 
            y_unlabeled_ground_truth=None,  
            beta_l=0.05, beta_u=0.95, 
            rho=0.1, gamma=0.05, theta=0.95, 
            max_iter=100):
        
        # 1. Preparación
        train_mask = df_labeled[split_col] == 'train'
        calib_mask = df_labeled[split_col] == 'calibration'
        test_mask = df_labeled[split_col] == 'test'
        
        X_train = df_labeled[train_mask].drop([target_col, split_col], axis=1)
        y_train = df_labeled[train_mask][target_col]
        
        X_calib = df_labeled[calib_mask].drop([target_col, split_col], axis=1)
        y_calib = df_labeled[calib_mask][target_col]
        
        X_test = df_labeled[test_mask].drop([target_col, split_col], axis=1)
        y_test = df_labeled[test_mask][target_col]
        
        # 2. FILTERING
        print("--- Etapa 1: Filtering ---")
        pool_rejects = filtering_step(X_train, X_unlabeled, beta_l, beta_u)
        
        # 3. WEAK LEARNER ESTÁTICO
        print("\n--- Etapa 2: Entrenando Weak Learner (Estático) ---")
        self.calibrated_weak_learner = self._fit_and_calibrate_once(X_train, y_train, X_calib, y_calib)
        
        # 4. STRONG LEARNER LÍNEA BASE (OPTIMIZADO)
        print("\n--- Etapa 3: Optimizando Strong Learner Base ---")
        self.strong_learner = self._optimize_strong_learner(X_train, y_train)
        self.best_model = self.strong_learner
        
        best_metric = bayesian_evaluation(self.strong_learner, X_test, y_test, pool_rejects, self.calibrated_weak_learner)
        print(f"Métrica Bayesiana Inicial: {best_metric:.4f}")
        
        X_augmented = X_train.copy()
        y_augmented = y_train.copy()
        
        # 5. CICLO SELF-LEARNING
        print("\n--- Etapa 4: Ciclo Self-Learning ---")
        for i in range(max_iter):
            print(f"\nIteración {i+1}:")
            
            if len(pool_rejects) == 0: break
                
            # A. Muestreo Aleatorio
            sample_size = int(len(pool_rejects) * rho)
            if sample_size < 10: sample_size = len(pool_rejects)
            
            candidates = pool_rejects.sample(n=sample_size, random_state=self.random_state + i)
            
            # Inspección de Realidad Oculta
            if y_unlabeled_ground_truth is not None:
                real_labels = y_unlabeled_ground_truth.loc[candidates.index]
                real_ones = real_labels.sum()
                real_zeros = len(real_labels) - real_ones
                print(f"  [REALIDAD OCULTA] Candidatos analizados ({len(candidates)}):")
                print(f"     -> Default Real (1): {real_ones} ({real_ones/len(candidates):.1%})")
                print(f"     -> Pago Real (0):    {real_zeros} ({real_zeros/len(candidates):.1%})")

            # B. Labeling Dinámico
            probs = self.calibrated_weak_learner.predict_proba(candidates)[:, 1]
            
            threshold_low = np.percentile(probs, gamma * 100)
            threshold_high = np.percentile(probs, theta * 100)
            
            idx_zeros = candidates.index[probs <= threshold_low]
            idx_ones = candidates.index[probs >= threshold_high]
            
            print(f"  [PREDICCIÓN MODELO]")
            print(f"     -> Umbral Bajo ({gamma*100}%): prob <= {threshold_low:.4f} => {len(idx_zeros)} etiquetados como 0")
            print(f"     -> Umbral Alto ({theta*100}%): prob >= {threshold_high:.4f} => {len(idx_ones)} etiquetados como 1")
            
            if len(idx_zeros) == 0 and len(idx_ones) == 0:
                print("  Ningún candidato seleccionado.")
                continue
            
            # C. Data Augmentation
            X_new = pd.concat([candidates.loc[idx_zeros], candidates.loc[idx_ones]])
            y_new = pd.concat([pd.Series(0, index=idx_zeros), pd.Series(1, index=idx_ones)])
            
            X_augmented = pd.concat([X_augmented, X_new])
            y_augmented = pd.concat([y_augmented, y_new])
            pool_rejects = pool_rejects.drop(X_new.index)
            
            # D. Re-Training Strong Learner (OPTIMIZADO)
            print("  Re-entrenando y optimizando Strong Learner con datos aumentados...")
            new_strong_learner = self._optimize_strong_learner(X_augmented, y_augmented)
            
            # E. Evaluación
            current_metric = bayesian_evaluation(new_strong_learner, X_test, y_test, pool_rejects, self.calibrated_weak_learner)
            print(f"  Métrica Bayesiana: {current_metric:.4f} (vs mejor: {best_metric:.4f})")
            
            if current_metric > best_metric:
                best_metric = current_metric
                self.best_model = new_strong_learner
                print("  -> Mejora confirmada.")
            else:
                print("  -> No hubo mejora. Early Stopping.")
                break
                
        return self.best_model
    
    def fit_with_param_search(
        self,
        df_labeled,
        X_unlabeled,
        target_col,
        split_col,
        param_grid,
        y_unlabeled_ground_truth=None,
        max_iter=100
    ):
        """
        Prueba múltiples combinaciones de parámetros del algoritmo
        (beta_l, beta_u, rho, gamma, theta) y retorna el mejor modelo.
        """

        results = []
        best_auc = -np.inf
        best_model = None
        best_params = None

        for params in ParameterGrid(param_grid):

            print("\n" + "="*60)
            print("Probando combinación:")
            print(params)

            model = BiasAwareSelfLearning(
                strong_learner_type=self.strong_learner_type,
                random_state=self.random_state,
                n_iter_search=self.n_iter_search,
                cv_folds=self.cv_folds
            )

            fitted_model = model.fit(
                df_labeled=df_labeled,
                X_unlabeled=X_unlabeled,
                target_col=target_col,
                split_col=split_col,
                y_unlabeled_ground_truth=y_unlabeled_ground_truth,
                beta_l=params["beta_l"],
                beta_u=params["beta_u"],
                rho=params["rho"],
                gamma=params["gamma"],
                theta=params["theta"],
                max_iter=max_iter
            )

            # evaluación final en test
            test_mask = df_labeled[split_col] == "test"
            X_test = df_labeled[test_mask].drop([target_col, split_col], axis=1)
            y_test = df_labeled[test_mask][target_col]

            preds = fitted_model.predict_proba(X_test)[:,1]
            auc = roc_auc_score(y_test, preds)

            print(f"AUC test: {auc:.4f}")

            results.append({
                **params,
                "auc": auc
            })

            if auc > best_auc:
                best_auc = auc
                best_model = fitted_model
                best_params = params

        results_df = pd.DataFrame(results).sort_values("auc", ascending=False)

        print("\n" + "="*60)
        print("MEJORES PARÁMETROS")
        print(best_params)
        print(f"MEJOR AUC: {best_auc:.4f}")
        print("="*60)

        self.best_model = best_model

        return best_model

# Ejecución del modelo Bias-Aware

## Validador Ingresos + Sin Historial

### Logit

In [ ]:
# ==============================================================================
# 1. EJECUCIÓN DEL MODELO BIAS-AWARE
# ==============================================================================
target_col = "default"
split_col = "split"

param_grid = {
    "beta_l": [0.01, 0.05, 0.025],
    "beta_u": [0.90, 0.95, 0.975],
    "rho": [0.05, 0.1],
    "gamma": [0.05, 0.10],
    "theta": [0.90, 0.95]
}


bias_aware_model = BiasAwareSelfLearning(
    strong_learner_type="logit",
    random_state=42
)

print("\n--- Entrenando Modelo Bias-Aware (Self-Learning) ---")
best_bias_model = bias_aware_model.fit_with_param_search(
    df_labeled=data_prepared_VI_SH["df_labeled"], 
    X_unlabeled=data_prepared_VI_SH["X_unlabeled"], 
    target_col=data_prepared_VI_SH["target_col"], 
    split_col=data_prepared_VI_SH["split_col"],
    param_grid=param_grid,
    max_iter=100
)


# ==============================================================================
# 2. COMPARACIÓN CON MODELO BASE
# ==============================================================================

# A. Preparar datos para evaluación final
# Extraemos el dataframe completo procesado
df_final = data_prepared_VI_SH["df_labeled"]
target_col = data_prepared_VI_SH["target_col"]
split_col = data_prepared_VI_SH["split_col"]

# Separamos Train (Solo aceptados originales) y Test (Holdout)
mask_train = df_final[split_col] == 'train'
mask_test = df_final[split_col] == 'test'

X_train_base = df_final[mask_train].drop([target_col, split_col], axis=1)
y_train_base = df_final[mask_train][target_col]

X_test_final = df_final[mask_test].drop([target_col, split_col], axis=1)
y_test_final = df_final[mask_test][target_col]

# B. Entrenar Modelo Base (Standard XGBoost)
# Usamos mismos hiperparámetros básicos para que la comparación sea justa
baseline_model = LogisticRegression(
    max_iter=5000,
    random_state=42
)

baseline_model.fit(X_train_base, y_train_base)

# C. Generar Predicciones
# Predicciones del Baseline
preds_base = baseline_model.predict_proba(X_test_final)[:, 1]

# Predicciones del Bias-Aware (usamos el modelo que nos devolvió el .fit())
preds_bias = best_bias_model.predict_proba(X_test_final)[:, 1]

# D. Calcular Métricas (AUC)
auc_base = roc_auc_score(y_test_final, preds_base)
auc_bias = roc_auc_score(y_test_final, preds_bias)

# E. Reporte de Resultados
print("\n" + "="*50)
print("RESULTADOS DE LA COMPARACIÓN (AUC en Test Set)")
print("="*50)
print(f"1. Base (Solo Aceptados):   {auc_base:.5f}")
print(f"2. Bias-Aware (Aceptados + Rechazados): {auc_bias:.5f}")
print("-" * 50)

mejora = (auc_bias - auc_base) * 100
if mejora > 0:
    print(f"¡ÉXITO! El método Bias-Aware mejoró el modelo en +{mejora:.2f} puntos porcentuales.")
else:
    print(f"El método Bias-Aware no superó al baseline ({mejora:.2f} pp). Intenta ajustar theta/gamma.")
print("="*50)

In [ ]:
'''
==================================================
RESULTADOS DE LA COMPARACIÓN (AUC en Test Set)
==================================================
1. Base (Solo Aceptados):   0.655
2. Bias-Aware (Aceptados + Rechazados): 0.657
--------------------------------------------------
'''

## Independientes + Quanto

### logit

In [ ]:
# ==============================================================================
# 1. EJECUCIÓN DEL MODELO BIAS-AWARE
# ==============================================================================
target_col = "default"
split_col = "split"

bias_aware_model = BiasAwareSelfLearning(
    strong_learner_type="logit",
    random_state=42
)
param_grid = {
    "beta_l": [0.01, 0.05, 0.025],
    "beta_u": [0.90, 0.95, 0.975],
    "rho": [0.05, 0.1],
    "gamma": [0.05, 0.10],
    "theta": [0.90, 0.95]
}



print("\n--- Entrenando Modelo Bias-Aware (Self-Learning) ---")
best_bias_model = bias_aware_model.fit_with_param_search(
    data_prepared_I_Q["df_labeled"], 
    data_prepared_I_Q["X_unlabeled"], 
    data_prepared_I_Q["target_col"], 
    data_prepared_I_Q["split_col"],
    param_grid=param_grid,
    max_iter=100
    )


# ==============================================================================
# 2. COMPARACIÓN CON MODELO BASE
# ==============================================================================

# A. Preparar datos para evaluación final
# Extraemos el dataframe completo procesado
df_final = data_prepared_I_Q["df_labeled"]
target_col = data_prepared_I_Q["target_col"]
split_col = data_prepared_I_Q["split_col"]

# Separamos Train (Solo aceptados originales) y Test (Holdout)
mask_train = df_final[split_col] == 'train'
mask_test = df_final[split_col] == 'test'

X_train_base = df_final[mask_train].drop([target_col, split_col], axis=1)
y_train_base = df_final[mask_train][target_col]

X_test_final = df_final[mask_test].drop([target_col, split_col], axis=1)
y_test_final = df_final[mask_test][target_col]

# B. Entrenar Modelo Base (Standard XGBoost)
# Usamos mismos hiperparámetros básicos para que la comparación sea justa
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

baseline_model.fit(X_train_base, y_train_base)

# C. Generar Predicciones
# Predicciones del Baseline
preds_base = baseline_model.predict_proba(X_test_final)[:, 1]

# Predicciones del Bias-Aware (usamos el modelo que nos devolvió el .fit())
preds_bias = best_bias_model.predict_proba(X_test_final)[:, 1]

# D. Calcular Métricas (AUC)
auc_base = roc_auc_score(y_test_final, preds_base)
auc_bias = roc_auc_score(y_test_final, preds_bias)

# E. Reporte de Resultados
print("\n" + "="*50)
print("RESULTADOS DE LA COMPARACIÓN (AUC en Test Set)")
print("="*50)
print(f"1. Base (Solo Aceptados):   {auc_base:.5f}")
print(f"2. Bias-Aware (Aceptados + Rechazados): {auc_bias:.5f}")
print("-" * 50)

mejora = (auc_bias - auc_base) * 100
if mejora > 0:
    print(f"¡ÉXITO! El método Bias-Aware mejoró el modelo en +{mejora:.2f} puntos porcentuales.")
else:
    print(f"El método Bias-Aware no superó al baseline ({mejora:.2f} pp). Intenta ajustar theta/gamma.")
print("="*50)

In [ ]:
'''
==================================================
RESULTADOS DE LA COMPARACIÓN (AUC en Test Set)
==================================================
1. Base (Solo Aceptados):   0.62094
2. Bias-Aware (Aceptados + Rechazados): 0.63030
--------------------------------------------------
'''